In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import warnings
import datetime
warnings.simplefilter("ignore", FutureWarning)

from demand_ninja.core import demand

curr_dir = pathlib.Path(".")
data_dir = curr_dir / "RTS-GMLC-master" / "RTS_Data"
load_data_dir = curr_dir / "load-data"

### Converting weather data to load data with DemandNinja

In [25]:
### Create load profiles; normalize to 2006 load data

# Read in list of buses
df_bus = pd.read_csv(curr_dir / "RTS-GMLC-master" / "RTS_Data" / "SourceData" / "bus.csv", index_col=[0])

for bus in df_bus.index:
    print(bus)
    skip_bus = False
    years = np.arange(1998, 2024)
    df_in_concat = pd.DataFrame()
    for year in years:
        fname = load_data_dir / "inputs" / f"bus{bus}" / f"NSRDB_weather-inputs_bus{bus}_{year}.csv"
        if not fname.exists():
            skip_bus = True
            continue
        df_in = pd.read_csv(fname, index_col=["datetime"])
        df_in.index = pd.to_datetime(df_in.index)

        # Get subset of columns needed for load calculation in demand ninja
        df_in = df_in[["Temperature", "Relative Humidity", "GHI", "Wind Speed"]]
        df_in.columns = ["temperature", "humidity", "radiation_global_horizontal", "wind_speed_2m"]
        df_in_concat = pd.concat([df_in_concat, df_in])
    df_in = df_in_concat.copy()

    if skip_bus:
        print("Skipping bus", bus)
        continue
    
    # Calculate demand with demand ninja
    df_out = demand(
        df_in,
        base_power=10,
        heating_power=0.8,
        cooling_power=1.2,
        cooling_threshold=16.0,
        heating_threshold=24.0,
        humidity_discomfort=0.003,
        solar_gains=0.01,
        smoothing=0.5,
    )["total_demand"]

    # Normalize by peak load in 2006
    peak_2006 = df_out.loc[df_out.index.year == 2006].max()
    df_out /= peak_2006

    # Save data
    # Profiles directory
    bus_dir = load_data_dir / "profiles" / f"bus{bus}"
    bus_dir.mkdir(exist_ok=True, parents=True)
    for year in years:
        df_out_year = df_out.loc[df_out.index.year == year]
        df_out_year.to_csv(bus_dir / f"NSRDB_load-profile_bus{bus}_{year}.csv")

101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325


In [ ]:
### DIAGNOSTICS / tuning the load model

In [2]:
# Read in list of buses
df_bus = pd.read_csv(curr_dir / "RTS-GMLC-master" / "RTS_Data" / "SourceData" / "bus.csv", index_col=[0])

In [3]:
# Seasons
summer = [6, 7, 8, 9]
winter = [11, 12, 1, 2]
shoulder = [10, 3, 4, 5]

In [6]:
# Read in NSRDB input data + convert to load data
diagnostics = False
for bus in df_bus.index:
    print(bus)
    skip_bus = False
    years = [2019, 2020, 2021, 2022, 2023]
    df_in_concat = pd.DataFrame()
    for year in years:
        fname = load_data_dir / "inputs" / f"bus{bus}" / f"NSRDB_weather-inputs_bus{bus}_{year}.csv"
        if not fname.exists():
            skip_bus = True
            continue
        df_in = pd.read_csv(fname, index_col=["datetime"])
        df_in.index = pd.to_datetime(df_in.index)

        # Get subset of columns needed for load calculation in demand ninja
        df_in = df_in[["Temperature", "Relative Humidity", "GHI", "Wind Speed"]]
        df_in.columns = ["temperature", "humidity", "radiation_global_horizontal", "wind_speed_2m"]
        df_in_concat = pd.concat([df_in_concat, df_in])
    df_in = df_in_concat.copy()

    if skip_bus:
        print("Skipping bus", bus)
        continue
    
    # Calculate demand with demand ninja
    df_out = demand(
        df_in,
        base_power=10,
        heating_power=0.8,
        cooling_power=1.2,
        cooling_threshold=16.0,
        heating_threshold=24.0,
        humidity_discomfort=0.003,
        solar_gains=0.01,
        smoothing=0.5,
    )["total_demand"]
    
    if diagnostics:
        # Get subset of Southwest demand for years
        df_sw = df_sw.loc[df_sw.index.year.isin(years)]
    
        # Plot
        df_out_norm = df_out / df_out.mean()
        df_sw_norm = df_sw / df_sw.mean()
        df_out_norm.plot()
        df_sw_norm.loc[df_in.index].plot(alpha=0.5)
        plt.show()
    
        plt.scatter(df_in["temperature"], df_out_norm, s=3, alpha=0.1)
        plt.scatter(df_in["temperature"], df_sw_norm.loc[df_in.index], s=3, alpha=0.1)
        plt.show()
    
        # Plot load shapes
        for season in [summer, winter, shoulder]:
            print(season)
            df_sw_season = df_sw_norm.loc[df_sw_norm.index.month.isin(season)]
            df_out_season = df_out_norm.loc[df_out_norm.index.month.isin(season)]
            df_sw_season_24hr = df_sw_season.groupby(df_sw_season.index.hour).mean()
            df_out_season_24hr = df_out_season.groupby(df_out_season.index.hour).mean()
    
            plot_sw = pd.concat([df_sw_season_24hr.iloc[8:], df_sw_season_24hr.iloc[0:8]])
            plot_out = df_out_season_24hr
    
            plt.plot(plot_out.values)
            plt.plot(plot_sw.values)
            plt.title("Load")
            plt.show()
    
            df_in_season = df_in.loc[df_in.index.month.isin(season)]
            df_in_season_24hr = df_in_season["temperature"].groupby(df_in_season.index.hour).mean()
            plt.plot(df_in_season_24hr)
            plt.title("Temp")
            plt.show()

101


In [7]:
df_out

datetime
2019-01-01 00:00:00    25.864341
2019-01-01 01:00:00    26.178853
2019-01-01 02:00:00    26.296219
2019-01-01 03:00:00    26.277020
2019-01-01 04:00:00    26.343696
                         ...    
2023-12-31 19:00:00    21.135610
2023-12-31 20:00:00    20.802080
2023-12-31 21:00:00    21.061270
2023-12-31 22:00:00    21.360289
2023-12-31 23:00:00    21.435298
Name: total_demand, Length: 43824, dtype: float64

### Downloading + processing EIA load data

In [ ]:
# Read in API key
with open("eia-api-key.txt", "r") as f:
    API_KEY = f.readlines()[0].strip()

In [ ]:
BASE_URL = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

RESPONDENTS = ["AZPS", "PNM", "SRP"]

START_DATE = "2016-01-01T00"
END_DATE   = "2018-12-31T00"

MAX_ROWS = 5000  # API row limit per request
DELAY = 0.2      # seconds between requests to avoid rate limits

def fetch_page(offset=0):
    """Fetch a single page of results from the EIA API."""
    params = {
        "api_key": API_KEY,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": RESPONDENTS,
        "start": START_DATE,
        "end": END_DATE,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": offset,
        "length": MAX_ROWS
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    data = response.json().get("response", {}).get("data", [])
    return data

def fetch_all_data():
    """Fetch all rows by paginating through the API."""
    all_data = []
    offset = 0
    
    while True:
        print(f"Fetching rows starting at offset {offset} ...")
        page = fetch_page(offset)
        if not page:
            break
        all_data.extend(page)
        if len(page) < MAX_ROWS:
            break
        offset += MAX_ROWS
        time.sleep(DELAY)
    
    print(f"Total rows fetched: {len(all_data)}")
    return all_data

def main():
    data = fetch_all_data()
    if not data:
        print("No data retrieved.")
        return
    
    df = pd.DataFrame(data)
    df['datetime'] = pd.to_datetime(df['period'])
    df = df.sort_values(['respondent', 'datetime'])
    
    # Save to CSV
    df.to_csv("EIA_hourly_load_2016_2018.csv", index=False)
    print("Saved to EIA_hourly_load_2016_2018.csv")

if __name__ == "__main__":
    main()

In [ ]:
# Read-in and organize load data
df = pd.read_csv(load_data_dir / "historical" / "EIA_hourly_load_2019_2025.csv")
df_load = df.loc[df["type-name"] == "Demand"]
df_load = df_load.pivot(index=["datetime"], columns=["respondent"], values=["value"])
df_load.columns = df_load.columns.droplevel(0)
df_load.index = pd.to_datetime(df_load.index)

In [ ]:
# Clean load data

df_load_clean = df_load.copy()
low_cutoffs = {col: 100 for col in df_load_clean.columns}
low_cutoffs["LDWP"] = 1000
low_cutoffs["WALC"] = 400
for col in df_load.columns:
    
    # Identify large percentage changes in data (indicating erroneous data)
    bwd_pct_change = (df_load_clean[col] + 1).pct_change(periods=1, fill_method=None).abs()
    fwd_pct_change = (df_load_clean[col] + 1).pct_change(periods=-1, fill_method=None).abs()
    pct_change = pd.concat([fwd_pct_change, bwd_pct_change], axis=1).max(axis=1)
    big_deltas = (pct_change > 0.2)
    
    # Identify where time series is below cutoff
    zeros = (df_load_clean[col] < low_cutoffs[col])

    # Flag index at these times
    flag_inds = df_load_clean[col].index[big_deltas | zeros]
    
    # Remove data at these times
    df_load_clean.loc[flag_inds, col] = np.nan
    
    # Identify large rolling z-scores
    window = 168
    rolling_mean = df_load_clean[col].rolling(window=window, min_periods=1, center=True).mean()
    rolling_std = df_load_clean[col].rolling(window=window, min_periods=1, center=True).std()
    rolling_zscore = (df_load_clean[col] - rolling_mean) / rolling_std
    big_zscores = (rolling_zscore > 3)

    # Flag index at these times
    flag_inds = df_load_clean[col].index[big_deltas | zeros | big_zscores]

     # Remove data at these times
    df_load_clean.loc[flag_inds, col] = np.nan

    # Interpolate data
    df_load_clean[col] = df_load_clean[col].interpolate(method="time")
    
    # Plot cleaned data
    df_load_clean[col].plot()
    plt.title(col)
    plt.show()

In [ ]:
# Calibrating Demand Ninja parameters with cleaned load data
summer = [5,6,7,8,9,10]
winter = [11,12,1,2,3,4]
shoulder = [4,11]

In [ ]:
# Get summer/winter load
df_sw = df_load_clean["SW"]
df_sw_summer = df_sw.loc[df_sw.index.month.isin(summer)]
df_sw_winter = df_sw.loc[df_sw.index.month.isin(winter)]

In [ ]:
# Get summer/winter peaks
summer_peaks = df_sw_summer.groupby(df_sw_summer.index.year).max()
winter_peaks = df_sw_winter.groupby(df_sw_winter.index.year).max()

In [ ]:
# Ratio of summer to winter peaks
summer_peaks / winter_peaks

In [ ]:
# Get shoulder season baseline demand
shoulder = [3,11]
df_sw_shoulder = df_sw.loc[df_sw.index.month.isin(shoulder)]
shoulder_mins = df_sw_shoulder.groupby(df_sw_shoulder.index.year).min()
summer_peaks / shoulder_mins